# GCP New models to be apply

In [2]:
ENDPOINT = "us-central1-aiplatform.googleapis.com"
REGION = "us-central1"
PROJECT_ID = "master-experiments-project"

In [3]:
import vertexai
from google.auth import default, transport

vertexai.init(project=PROJECT_ID, location=REGION)

In [28]:
credentials, _ = default()
auth_request = transport.requests.Request()
credentials.refresh(auth_request)

In [21]:
ENDPOINT = "us-central1-aiplatform.googleapis.com"
REGION = "us-central1"
PROJECT_ID = "master-experiments-project"

In [30]:
from typing import Any, Dict, List, Optional
import openai
from langchain_core.language_models import BaseLLM
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.pydantic_v1 import Field, root_validator


class GoogleGardenLLM(BaseLLM):
    """
    Custom LLM class for interacting with Google Garden Models API using the OpenAI client.
    """

    project_id: str = Field(..., description="Google Cloud Project ID")
    location: str = Field(default="us-central1", description="Google Cloud Location")
    model_id: str = Field(
        ..., description="Model ID (e.g., meta/llama-3.2-90b-vision-instruct-maas)"
    )
    max_tokens: int = Field(
        default=4096, description="Maximum number of tokens to generate"
    )
    credentials: Any = Field(..., description="Google Cloud credentials")

    @root_validator
    def validate_environment(cls, values: Dict) -> Dict:
        """Validate that the Google Cloud credentials are set."""
        if not values.get("credentials"):
            raise ValueError("Google Cloud credentials must be provided.")
        return values

    def _call(
        self,
        prompt: str,
        image_url: Optional[str] = None,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        """
        Call the Google Garden Models API with the given prompt and optional image URL.
        """
        # Initialize the OpenAI client
        client = openai.OpenAI(
            base_url=f"https://{self.location}-aiplatform.googleapis.com/v1beta1/projects/{self.project_id}/locations/{self.location}/endpoints/openapi",
            api_key=self.credentials.token,
        )

        # Prepare the messages
        messages = [{"role": "user", "content": [{"text": prompt, "type": "text"}]}]
        if image_url:
            messages[0]["content"].insert(
                0, {"image_url": {"url": image_url}, "type": "image_url"}
            )

        # Make the API request
        response = client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            max_tokens=self.max_tokens,
        )

        # Extract and return the response
        return response.choices[0].message.content

    def _generate(
        self,
        prompt: str,
        image_url: Optional[str] = None,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        """
        Call the Google Garden Models API with the given prompt and optional image URL.
        """
        # Initialize the OpenAI client
        client = openai.OpenAI(
            base_url=f"https://{self.location}-aiplatform.googleapis.com/v1beta1/projects/{self.project_id}/locations/{self.location}/endpoints/openapi",
            api_key=self.credentials.token,
        )

        # Prepare the messages
        messages = [{"role": "user", "content": [{"text": prompt, "type": "text"}]}]
        if image_url:
            messages[0]["content"].insert(
                0, {"image_url": {"url": image_url}, "type": "image_url"}
            )

        # Make the API request
        response = client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            max_tokens=self.max_tokens,
        )

        # Extract and return the response
        return response.choices[0].message.content

    @property
    def _llm_type(self) -> str:
        """Return the type of LLM."""
        return "google_garden_llm"

In [31]:
# Initialize the custom LLM
from google.oauth2 import service_account

# Load Google Cloud credentials
llm = GoogleGardenLLM(
    project_id="your-project-id",
    location="us-central1",
    model_id="meta/llama-3.2-90b-vision-instruct-maas",
    max_tokens=4096,
    credentials=credentials,
)

# # Use the LLM in a LangChain chain
# from langchain.chains import LLMChain
# from langchain.prompts import PromptTemplate

# prompt_template = PromptTemplate(
#     input_variables=["question"],
#     template="Answer the following question: {question}"
# )

# chain = LLMChain(llm=llm, prompt=prompt_template)

# # Run the chain
# response = chain.run("What’s in this image?")
# print(response)

In [33]:
llm.invoke("tell me a joke")

TypeError: Client.__init__() got an unexpected keyword argument 'proxies'